# Lab type: review
# Course: ML401 — MLOps & Model Deployment
# Lesson: Training-Serving Skew
# Task: A working feature pipeline is provided. Review it for skew risk: identify where skew could enter, how you would detect it in production, and what changes would make it skew-resistant.

In [ ]:
# !pip install scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import joblib

np.random.seed(42)
n = 5000

# Simulated training dataset (batch export from data warehouse)
training_data = pd.DataFrame({
    'tenure_days': np.random.randint(1, 2000, n),
    'monthly_spend': np.random.uniform(20, 500, n),
    'support_tickets_30d': np.random.poisson(1.5, n),
    'plan_type': np.random.choice(['basic', 'pro', 'enterprise'], n, p=[0.5, 0.35, 0.15]),
    'days_since_last_login': np.random.exponential(7, n),
})

# Label: churned within 90 days
churn_prob = (
    0.3 * (training_data['days_since_last_login'] > 14).astype(float) +
    0.2 * (training_data['support_tickets_30d'] > 3).astype(float) +
    0.1 * (training_data['monthly_spend'] < 50).astype(float)
)
training_data['churned'] = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

print(f'Dataset shape: {training_data.shape}')
print(f'Churn rate: {training_data["churned"].mean():.2%}')
training_data.head()

In [ ]:
# Training pipeline
X = training_data.drop('churned', axis=1)
y = training_data['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['tenure_days', 'monthly_spend', 'support_tickets_30d', 'days_since_last_login']
categorical_features = ['plan_type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

model_pipeline.fit(X_train, y_train)

test_auc = roc_auc_score(y_test, model_pipeline.predict_proba(X_test)[:, 1])
print(f'Test AUC: {test_auc:.4f}')

joblib.dump(model_pipeline, 'churn_pipeline.joblib')
print('Pipeline saved.')

## Review questions

The pipeline above is **correctly implemented** — there is no data leakage and the full pipeline is saved.

However, in production, this model will receive features from a **live event stream API** rather than from the same batch export used for training.

Answer the following questions based on your review of the pipeline and the production scenario.

### Question 1: Skew entry points

For each feature below, describe one realistic production scenario where the feature value at inference time could differ from the feature value at training time, and explain whether the pipeline would detect the difference.

| Feature | Skew scenario | Would the pipeline detect it? |
|---------|--------------|-------------------------------|
| `support_tickets_30d` | | |
| `days_since_last_login` | | |
| `plan_type` | | |

<details>
<summary>🔑 Reveal answer — Question 1</summary>

| Feature | Skew scenario | Would the pipeline detect it? |
|---------|--------------|-------------------------------|
| `support_tickets_30d` | The live API counts tickets opened in the last 7 days; training used a 30-day window from the warehouse. The value is numerically valid and in range. | **No.** The scaler receives a valid float and transforms it silently. The feature represents a different signal. |
| `days_since_last_login` | The API returns `null` for users who have never logged in; training had no nulls. Alternatively, the reference timestamp shifts (e.g., last app login vs. last API call). | **Partially.** A `null` raises a `ValueError` at predict time (explicit failure). A wrong reference point produces a valid-looking number — the pipeline cannot distinguish it from a correct value. |
| `plan_type` | The company launches a new plan tier (`'starter'`) after deployment. | **No error.** `OrdinalEncoder` encodes the unknown value as `-1` and prediction continues. The pipeline does not log a warning or flag the anomaly. |

</details>

### Question 2: The OrdinalEncoder unknown value

The pipeline uses `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)` for `plan_type`.

If the company launches a new plan type called `'starter'` after this model is deployed, what value will the pipeline encode it as? What will the GradientBoostingClassifier do with this value? Is this the correct behaviour for your use case?

Write your answer below:

In [ ]:
# Demonstrate what happens with an unknown plan type
loaded_pipeline = joblib.load('churn_pipeline.joblib')

new_customer = pd.DataFrame([{
    'tenure_days': 45,
    'monthly_spend': 89.99,
    'support_tickets_30d': 0,
    'plan_type': 'starter',  # new plan type not seen at training time
    'days_since_last_login': 2.5
}])

# What does the pipeline predict?
prob = loaded_pipeline.predict_proba(new_customer)[0, 1]
print(f'Churn probability for unknown plan type: {prob:.4f}')

# What encoding does it receive?
enc = loaded_pipeline.named_steps['preprocessor'].named_transformers_['cat']
encoded = enc.transform(new_customer[['plan_type']])
print(f'Encoded plan_type value: {encoded[0, 0]}')

<details>
<summary>🔑 Reveal answer — Question 2</summary>

**What value is encoded:** `'starter'` maps to `-1` — the configured `unknown_value`.

**What the classifier does:** `GradientBoostingClassifier` treats `-1` as a valid numeric feature value. In a decision tree, every split compares the value against a threshold learned from `{0, 1, 2}` (basic/enterprise/pro). Since `-1` is below all of them, every split routes `starter` left — meaning the model always behaves as if `starter` is "lower than basic." No error is raised.

**Is this correct?** Only if `starter` is genuinely the lowest-tier plan and the classifier's ordinal ordering (`basic < enterprise < pro` or whatever `OrdinalEncoder` learned) happens to match the business logic. In practice this is a silent assumption. A safer approach: detect the unknown category before prediction, log a warning, and either return a default score or raise a handled exception so the caller can decide.

</details>

### Question 3: Detection strategy

Propose a monitoring strategy for this pipeline. For each type of skew you identified in Question 1, specify:
- The monitoring signal you would track
- The reference baseline (what does 'normal' look like?)
- The alert threshold you would set
- What action you would take when the alert fires

Write your strategy below:

<details>
<summary>🔑 Reveal answer — Question 3</summary>

**Numeric feature drift (`support_tickets_30d`, `days_since_last_login`, `monthly_spend`, `tenure_days`):**
- *Signal:* Population Stability Index (PSI) computed daily against the training distribution, or a rolling z-score of the daily mean.
- *Baseline:* Training-time statistics (mean, std, percentile buckets) saved in the contract file.
- *Threshold:* PSI > 0.2 (major shift) → alert; PSI 0.1–0.2 → warning.
- *Action:* Investigate upstream data pipeline for window/aggregation changes; if drift persists >3 consecutive days, trigger a scheduled retrain.

**Categorical feature drift (`plan_type`):**
- *Signal:* Daily frequency of each known category; rate of `unknown_value = -1` hits.
- *Baseline:* Training-time `value_counts` proportions.
- *Threshold:* Unknown-category rate > 1% of daily requests → alert.
- *Action:* Add the new category to the encoder, retrain, and redeploy as a new versioned artefact.

**Model output monitoring:**
- Track daily mean predicted churn probability vs. training-time base rate (≈12%). Alert on >5 pp absolute drift sustained over 3 days. This catches skew that slips past feature checks.

</details>

### Question 4: Making it skew-resistant

The current pipeline saves the model artefact but does not save the training data statistics or feature distribution baseline.

Modify the training code below to also save:
1. The expected feature names and their training-time statistics (mean, std for numeric; value_counts for categorical)
2. A function that validates an inference request against these statistics before prediction

Hint: the `ColumnTransformer` contains the fitted scalers; you can extract their parameters.

In [ ]:
import json

# Your implementation here:
# 1. Extract training statistics from the fitted pipeline
# 2. Save them to a JSON contract file
# 3. Write a validate_request(X, contract) function that warns when a feature
#    is outside expected bounds

scaler = model_pipeline.named_steps['preprocessor'].named_transformers_['num']

# Extract the scaler statistics
training_stats = {
    'numeric_features': {
        feat: {'mean': float(mean), 'std': float(std)}
        for feat, mean, std in zip(numeric_features, scaler.mean_, scaler.scale_)
    },
    'categorical_features': {
        'plan_type': list(training_data['plan_type'].value_counts().index)
    }
}

print(json.dumps(training_stats, indent=2))

# TODO: write validate_request(X_inference, training_stats) that:
# - Checks all expected features are present
# - Warns if any numeric feature is more than 3 std deviations from training mean
# - Warns if any categorical feature has an unknown value

<details>
<summary>🔑 Reveal answer — Question 4</summary>

**Extracting statistics from the fitted pipeline:**
The `StandardScaler` is at `model_pipeline.named_steps['preprocessor'].named_transformers_['num']`. Its `mean_` and `scale_` arrays align with the `numeric_features` list. The `OrdinalEncoder` is at `named_transformers_['cat']`; its `categories_[0]` lists the known plan-type strings.

**`validate_request` logic:**
1. Assert all expected column names are present in the incoming DataFrame — raises `ValueError` on a missing feature.
2. For each numeric feature: compute `abs(value - mean) / std`. If > 3, append a warning string (do not raise — let the prediction proceed but log the anomaly).
3. For `plan_type`: check whether each incoming value is in the saved `categories_` list. Append a warning for any unknown value.
4. Return `(warnings: list[str])` so the caller can decide whether to block, log, or route to a fallback model.

**Why return warnings instead of raising:** A hard raise on every out-of-distribution request would block production traffic. Soft warnings allow the model to serve while alerting the monitoring system.

</details>

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Skew entry points:** Each feature can diverge silently between training batch and live API — different aggregation windows, shifted reference timestamps, or new categorical values — and none trigger a pipeline error.

2. **OrdinalEncoder unknown handling:** Unknown categories encode as `-1`; tree-based models treat this as a valid split value, silently routing unknowns to the lowest-ordinal branch.

3. **Detection strategy:** Monitor feature distributions daily (PSI or z-score) against saved training statistics, plus unknown-category rate for categoricals and output score drift; alert at defined thresholds and retrain if drift persists.

4. **Skew-resistant design:** Serialise `StandardScaler.mean_`/`scale_` and `OrdinalEncoder.categories_` into a contract file; add a `validate_request` function that checks feature presence, numeric bounds (3σ), and categorical membership before every prediction, returning soft warnings rather than hard errors.

</details>